# is-differentiable-flag — ex2: wire is_differentiable into a Recipe-building wrap_forward_fn

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `is-differentiable-flag`. Running the final beacon cell reports progress against the `Backprop: is_differentiable flag` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: is_differentiable flag` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`is-differentiable-flag`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "is-differentiable-flag"
DD_SUBTOPIC = "Backprop: is_differentiable flag"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## is_differentiable wired into wrap_forward_fn — quick refresher

ex1 isolated `make_check_requires_grad(is_differentiable)` as a stand-alone factory. The ex2 facet is the **integration** path: the same flag is also the gate for ATTACHING A RECIPE inside `wrap_forward_fn`.

Two distinct effects of `is_differentiable=False`, both flowing from the same flag:
- Output `requires_grad` is forced False (the three-gate AND short-circuits on gate 2).
- Output `recipe` is `None` — backprop treats the node as a leaf and stops there, exactly as if the user had detached.

Both effects are necessary: setting `requires_grad=False` without skipping the Recipe would leave a dangling parent edge that the reverse pass might still try to walk.

### Exercise 2 — wire is_differentiable into a Recipe-building wrap_forward_fn

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the per-op is_differentiable flag as a TWO-effect gate inside wrap_forward_fn: it forces requires_grad=False AND skips Recipe construction, both via the same closure-captured boolean.
> Keywords: is-differentiable, recipe-gate, wrap-forward-fn, closure
> ```

**KCs targeted:** `is-differentiable-flag`, `non-diff-fn-wrap`

Implement `wrap_forward_fn(fwd_fn, is_differentiable=True)`. Different surface from ex1's `make_check_requires_grad` factory: this is the FULL wrapper that produces a working `tensor_func`.

Requirements:

1. **Unbox** MiniTensor args to their raw arrays. Non-tensor args pass through.
2. **Forward call** `fwd_fn(*raw_args, **kwargs)`.
3. **Three-gate AND** to compute `requires_grad`:
   `grad_tracking_enabled AND is_differentiable AND any-tracked-input`.
4. **Box** the result as a `MiniTensor(out_arr, requires_grad)`.
5. **Conditional Recipe.** ATTACH a Recipe ONLY when `requires_grad` is True — both effects of `is_differentiable=False` flow from this same boolean.

Verify the two effects co-occur:
- A differentiable op (`add` wrapper) with tracked inputs produces `requires_grad=True` AND a populated `recipe`.
- A non-differentiable op (`eq` wrapper) with tracked inputs produces `requires_grad=False` AND `recipe=None`.
- The closure-captured flag is sticky: re-using the SAME wrapped op across many calls keeps the flag's effect consistent.

Setup cell provides `MiniTensor`, `Recipe`, `grad_tracking_enabled=True`. Don't call `torch.autograd`.

In [ ]:
def wrap_forward_fn(fwd_fn, is_differentiable: bool = True):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_arr = fwd_fn(*raw_args, **kwargs)
        requires_grad = (
            globals()['grad_tracking_enabled']
            and is_differentiable
            and any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
        )
        out = MiniTensor(out_arr, requires_grad)
        if requires_grad:
            parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func


<details><summary>Solution</summary>

```python
def wrap_forward_fn(fwd_fn, is_differentiable: bool = True):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_arr = fwd_fn(*raw_args, **kwargs)
        requires_grad = (
            globals()['grad_tracking_enabled']
            and is_differentiable
            and any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
        )
        out = MiniTensor(out_arr, requires_grad)
        if requires_grad:
            parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func
```

**Why BOTH effects flow from the same flag.** The `if requires_grad:` guard around Recipe construction makes the two effects atomic. If you build the Recipe unconditionally and only set `out.requires_grad = False`, the reverse pass — which keys on `recipe is not None` — would still walk into the non-diff op's parents and crash on the missing back fn.

**Two wrappers, same fwd fn, different flags.** This is uncommon in production (you'd just register `t.add` once with the right flag), but it pins down the closure semantics: each call to `wrap_forward_fn` creates a SEPARATE closure with its own captured `is_differentiable`. Reused references to the same `fwd_fn` do not share state.

**Reading the global via `globals()`.** A bare `grad_tracking_enabled` reference inside `tensor_func` ALSO works (Python resolves it via LEGB → module globals at call time), but `globals()['...']` is more explicit when the wrapper is later moved into a class or another module.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()